### IMPORTAMOS LAS LIBRERÍAS NECESARIAS

In [1]:
import random
import time
import numpy as np
from IPython.display import clear_output
import sys

### DEFINICIÓN DE LA CLASE PARA JUGAR

In [7]:
class hexapawn:
    def __init__(self, seed=27779):
        self.seed = int(seed)
        self.gen = random.Random(self.seed)

    #Regresa un número entero aleatorio
    def random_int(self):
        return self.gen.randint(1, 1000)
    
    #Coloca el tablero para un juego desde 0
    def poner_el_juego(self, n, turno=0):
        primera_fila=[2 for i in range(n)]
        ultima_fila=[1 for i in range(n)]
        fila_medio=[0 for i in range(n)]

        tablero=[]
        tablero.append(primera_fila.copy())
        for i in range(n-2): tablero.append(fila_medio.copy())
        tablero.append(ultima_fila.copy())

        return tablero, 1 if turno==0 else turno
    
    #Imprime de manera bonita el estado del juego, junto con el turno
    def ver_tablero(self, tablero, turno=1):
        #clear_output()
        count=0
        print("\n╔", end="")
        for i in range(len(tablero)*4-1):
            print("═", end="")
            count+=1
            if(count==100):
                return 
            
        print("╗   filas     turno de: ", "X" if turno==1 else "O")
        
        for i in range(len(tablero)):
            print("║", end="")

            for j in range(len(tablero[1])):
                print("   " if tablero[i][j]==0 else " X " if tablero[i][j]==1 else " O ", end="")
                if j!=len(tablero[1])-1: print("│", end="")
                
            print("║ ", i)
            
            if i!=len(tablero)-1:
                print("║", end="")
                for j in range(len(tablero[1])):
                    for k in range(3):
                        print("-", end="")
                        
                    if j!=len(tablero[1])-1:
                        print("┼", end="")
                print("║")
        
        print("╚", end="")
        for i in range(len(tablero[1])*4-1):
            print("═", end="")

        print("╝\n ", end="")
        for i in range(len(tablero[1])):
            print(f" {i}  ", end="")
            
        print("\n\ncolumnas\n\n")
        
    #Cambia de turno para el siguiente
    def siguiente_turno(self, turno):
        return 1 if turno==2 else 1
    
    #Filas original, columnas original, filas terminal, columnas terminal
    #te dice si el tiro que quieres hacer es legal o no
    def tiro_legal(self, f_o, c_o, f_t, c_t, tablero):
        if(f_t<0 or f_t>len(tablero)-1 or c_t<0 or c_t>len(tablero)-1):
            return False
        
        if(tablero[f_o][c_o]==1):
            return (((f_t==f_o-1) and (c_o==c_t)) and (tablero[f_t][c_t]==0)) or (((f_t==f_o-1) and (c_o==c_t-1)) and (tablero[f_t][c_t]==2)) or (((f_t==f_o-1) and (c_o==c_t+1)) and (tablero[f_t][c_t]==2))
        
        elif(tablero[f_o][c_o]==2):
            return (((f_t==f_o+1) and (c_o==c_t)) and (tablero[f_t][c_t]==0)) or (((f_t==f_o+1) and (c_o==c_t-1)) and (tablero[f_t][c_t]==1)) or (((f_t==f_o+1) and (c_o==c_t+1)) and (tablero[f_t][c_t]==1));
        
        else:
            return False
    
    #recibe un tablero junto con el turno y da un tiro aleatorio, regresa el tablero con el
    #tiro regristrado y el siguiente turno en una tupla
    def tiro_random(self, tablero, turno):
        #posibles almacena un vector, de coordenadas de todas las fichas del jugador del que queremos hacer el tiro
        posibles=[]

        for i in range(len(tablero)):
            for j in range(len(tablero[0])):
                if(tablero[i][j]==turno):
                    posibles.append([i, j])

        self.gen.shuffle(posibles)
        
        while posibles:
            direcciones=[-1,0,1]
            elegido=posibles[len(posibles)-1]
            self.gen.shuffle(direcciones)
            
            while direcciones:
                if(self.tiro_legal(elegido[0],
                                   elegido[1],
                                   elegido[0]+(-1 if turno==1 else 1),
                                   elegido[1]+direcciones[len(direcciones)-1],
                                   tablero)):
                    tablero[elegido[0]][elegido[1]]=0
                    tablero[elegido[0]-1 if turno==1 else elegido[0]+1][elegido[1]+direcciones[len(direcciones)-1]]=turno

                    return (tablero, 2 if turno==1 else 1)
                
                else:
                    direcciones.pop()
                
            
            posibles.pop()
        
        return tablero, 0
    
    def todos_los_tiros(self, tablero, turno):
        posibles=[]

        todos=[]
        tablero_copia=[fila.copy() for fila in tablero]

        for i in range(len(tablero)):
            for j in range(len(tablero[0])):
                if(tablero[i][j]==turno):
                    posibles.append([i, j])

        self.gen.shuffle(posibles)

        while(posibles):
            direcciones=[-1, 0, 1]
            elegido=posibles[-1]

            self.gen.shuffle(direcciones)
            
            while(direcciones):
                if(self.tiro_legal(elegido[0],
                                   elegido[1],
                                   elegido[0]+(-1 if turno==1 else 1),
                                   elegido[1]+direcciones[-1],
                                   tablero)):
                    
                    tablero[elegido[0]][elegido[1]]=0
                    
                    tablero[elegido[0]+(-1 if turno==1 else 1)][elegido[1]+direcciones[-1]]=turno

                    todos.append([fila.copy() for fila in tablero])
                    tablero=[fila.copy() for fila in tablero_copia]
                    
                    direcciones.pop()
                
                else:
                    direcciones.pop()
                
            posibles.pop()

        return todos, turno

    #condicion 1 de ganar: llegar al otro lado, regresa -1 si nadie satisface esta 
    #condición, de lo contrario, regresa la ficha ganadora
    def ganar_1(self, tablero):
        for i in range(0, len(tablero), len(tablero)-1):
            for j in range(len(tablero[0])):
                if i==0:
                    if(tablero[i][j]==1):
                        return True
                else:
                    if(tablero[i][j]==2):
                        return True
        return False
    
    #condicion 2 de ganar: uno de los dos ya no tiene fichas, regresa -1 si nadie satisface, 
    #de lo contrario regresa la ficha ganadora
    
    #creo que aquí hay un problema, no siempre detecta cuando ya no pueden seguir los tiros
    #al menos creo que esa es la razón de un problema que surgió al generar los tiros aleatorios
    #que para terminar los tiros pide un tablero con finalizacion, pero no se detectó por parte de esta funcion
    #pero ese fue un solo caso particular, en los demás si sigue funcionando
    def ganar_2(self, tablero):
        conteo=[0, 0]
        for i in range(len(tablero)):
            for j in range(len(tablero[0])):
                if(tablero[i][j]!=0):
                    conteo[tablero[i][j]-1]+=1
                    
        return conteo[0]==0 or conteo[1]==0

    #condicion 3 de ganar: ya nadie puede tirar. regresa 1 si esta condicion se cumple,
    #regresa 0 de lo contrario
    def ganar_3(self, tablero):
        fichas=[]

        for i in range(len(tablero)):
            for j in range(len(tablero[0])):
                if(tablero[i][j])!=0:
                    fichas.append([i, j])

        for i in range(len(fichas)):
            direcciones=[-1, 0, 1]
            for j in range(3):
                if(self.tiro_legal(fichas[i][0],
                                   fichas[i][1],
                                   fichas[i][0]+(-1 if tablero[fichas[i][0]][fichas[i][1]]==1 else 1),
                                   fichas[i][1]+direcciones[j],
                                   tablero)):
                    return False

        return True
    
    #verifica las condiciones de terminado del juego
    def condiciones_de_terminado(self, tablero):
        return self.ganar_1(tablero) or self.ganar_2(tablero) or self.ganar_3(tablero)
    
    #le das un tablero y turno, y termina el juego con tiros aleatorios. Regresa 
    #la ficha ganadora. Adicionalmente, puedes ver qué onda con el juego
    def terminar_juego(self, tablero, turno, verbose=0):
        n=0
        while(1):
            tablero, turno=self.tiro_random(tablero, turno)
            if(verbose):
                self.ver_tablero(tablero,turno)

            if(self.condiciones_de_terminado(tablero) or turno==0):
                if(verbose):
                    if(turno==1):
                        print("VICTORIA DE O")
                    else:
                        print("VICTORIA DE O")
                        
                return 2 if turno==1 else 1
            
        return 0
    
    #te da un juego contra humano dado un tamaño de tablero, no regresa nada, solo es
    #para entretenimiento
    def juego_random_contra_humano(self, tamano):
        print("JUEGO CONTRA HUMANO:")
        tablero, turno=self.poner_el_juego(tamano)
        
        seguir=1
        self.ver_tablero(tablero,turno)

        while(seguir==1):
            while(1):
                f_o=int(input("\nFilas(origen):"))
                c_o=int(input("\nColumnas(origen):"))
                f_t=int(input("\nFilas(terminal):"))
                c_t=int(input("\nColumnas(terminal):"))
                if(self.tiro_legal(f_o, 
                                   c_o,
                                   f_t,
                                   c_t,
                                   tablero)==True):
                    tablero[f_o][c_o]=0
                    tablero[f_t][c_t]=1
                    break
                else:
                    print("\nTiro no permitido, intenta de nuevo.")

            if(turno==1):
                turno=2
            else:
                turno=1

            self.ver_tablero(tablero,turno)

            if(self.condiciones_de_terminado(tablero)):
                print("\nVICTORIA DE:", "O\n\n\nQuieres seguir?:(1=si, 0=no)" if turno==1 else "X\n\n\nQuieres seguir?:(1=si, 0=no)")
                seguir=int(input())
                if(seguir!=0):
                    tablero, turno=self.poner_el_juego(tamano)
                    self.ver_tablero(tablero,turno)
                
                else:
                    print("\nGracias por jugar, suerte para la próxima! :)" if turno==1 else "\nGracias por jugar, mejoraré para ganarte la próxima! >:)")
            
            else:
                print("\ntiro de maquina:")
                tablero, turno=self.tiro_random(tablero,turno)
                self.ver_tablero(tablero,turno)
                if(self.condiciones_de_terminado(tablero)):
                    print("\nVICTORIA DE:", "O\n\n\nQuieres seguir?:(1=si, 0=no)" if turno==1 else "X\n\n\nQuieres seguir?:(1=si, 0=no)")
                    seguir=int(input())
                    if(seguir!=0):
                        tablero, turno=self.poner_el_juego(tamano)
                        self.ver_tablero(tablero,turno)
                    
                    else:
                        print("\nGracias por jugar, suerte para la próxima! :)" if turno==1 else "\nGracias por jugar, mejoraré para ganarte la próxima! >:)")
    
if __name__=="__main__":
    juego=hexapawn()
    
    juego.juego_random_contra_humano(3)

JUEGO CONTRA HUMANO:

╔═══════════╗   filas     turno de:  X
║ O │ O │ O ║  0
║---┼---┼---║
║   │   │   ║  1
║---┼---┼---║
║ X │ X │ X ║  2
╚═══════════╝
  0   1   2  

columnas



╔═══════════╗   filas     turno de:  O
║ O │ O │ O ║  0
║---┼---┼---║
║   │   │ X ║  1
║---┼---┼---║
║ X │ X │   ║  2
╚═══════════╝
  0   1   2  

columnas



tiro de maquina:

╔═══════════╗   filas     turno de:  X
║   │ O │ O ║  0
║---┼---┼---║
║ O │   │ X ║  1
║---┼---┼---║
║ X │ X │   ║  2
╚═══════════╝
  0   1   2  

columnas



╔═══════════╗   filas     turno de:  O
║   │ O │ O ║  0
║---┼---┼---║
║ X │   │ X ║  1
║---┼---┼---║
║ X │   │   ║  2
╚═══════════╝
  0   1   2  

columnas



tiro de maquina:

╔═══════════╗   filas     turno de:  X
║   │   │ O ║  0
║---┼---┼---║
║ O │   │ X ║  1
║---┼---┼---║
║ X │   │   ║  2
╚═══════════╝
  0   1   2  

columnas



VICTORIA DE: O


Quieres seguir?:(1=si, 0=no)

Gracias por jugar, suerte para la próxima! :)


### DEFINICIÓN DE NODOS DEL ARBOL

In [3]:
class nodo:
    """
    Clase para representar los nodos del árbol.
    """

    def __init__(self, tablero, turno, g_p):
        #estructuras de navegación
        self.pad=None            # referencia al nodo padre
        self.hijos = []          # lista de referencias a los nodos hijos

        #datos contenida
        self.tablero = tablero   # tablero del juego específico en el nodo
        self.g_p = g_p           # 1: ganador, -1: perdedor, 0: sigue el juego
        self.uct = 6.0           # calificación asignada al tablero
        self.n = 0               # cantidad de veces que se ha visitado el nodo
        self.wins = 0            # cantidad de victorias del subárbol
        self.turno = turno       # turno del siguiente tiro

### ESTRUCTURA DEL ARBOL

In [6]:
class arbol:
    def __init__(self, clase_juego, seed=27779):
        self.seed=int(seed)
        self.gen=random.Random(self.seed)
        self.juego=clase_juego
        self.raiz=None
        self.victoria_humana=0
        self.victoria_maquina=0

    #Regresa un número entero aleatorio
    def random_int(self):
        return self.gen.randint(1, 1000)
    
    #Regresa un número real aleatorio entre 0 y 1
    def random_real(self):
        return self.gen.random()
    
    def insertar(self, padr, tablero_, turno_, g_p_):
        if(padr!=None):
            hijo=nodo(tablero_, turno_, g_p_)
            hijo.pad=padr
            padr.hijos.append(hijo)
            return hijo
        
        else:
            return None
        
    #RECORRIDOS
    def postorden(self, actual):
        if(actual!=None):
            for i in range(len(actual.hijos)):
                self.postorden(actual.hijos[i])

            print()

            for i in range(len(actual.tablero)):
                for j in range(len(actual.tablero[0])):
                    if(actual.tablero[i][j]==0):
                        print(" ", end="")
                    
                    elif(actual.tablero[i][j]==1):
                        print("X", end="")
                        
                    else:
                        print("O", end="")
                    
                print()
            
    def preorden(self, actual):
        if(actual!=None):
            print()
            for i in range(len(actual.tablero)):
                for j in range(len(actual.tablero[0])):
                    if(actual.tablero[i][j]==0):
                        print(" ", end="")
                    
                    elif(actual.tablero[i][j]==1):
                        print("X", end="")
                    
                    else:
                        print("O", end="")

                print()

            print("turno: ", actual.turno, "g_p: ", actual.g_p)

            for i in range(len(actual.hijos)):
                self.preorden(actual.hijos[i])
                
    def fancy_print(self, actual, indentacion=0):
        if(actual!=None):
            for i in range(len(actual.hijos)):
                self.fancy_print(actual.hijos[i], indentacion+7)

            for i in range(len(actual.tablero)):
                for k in range(indentacion):
                    print(" ", end="")

                for j in range(len(actual.tablero[0])):
                    if(actual.tablero[i][j]==0):
                        print(" ", end="")
                    elif(actual.tablero[i][j]==1):
                        print("X", end="")
                    else:
                        print("O", end="")

                print()

            for k in range(indentacion):
                print(" ", end="")

            print("trn ", actual.turno, " g/p ", actual.g_p, " uct ", actual.uct, " vsts ", actual.n, " wns ", actual.wins, "\n")

    def calcular_uct(self, nodo, constante=2):
        if(nodo!=None):
            if(nodo.pad!=None):
                if(nodo.n!=0):
                    nodo.uct=((nodo.wins*1.0)/(nodo.n*1.0))+np.sqrt((constante*np.log(nodo.pad.n))/(nodo.n*1.0))
                
                else:
                    nodo.uct=6.0
                    
            else:
                if(nodo.n!=0):
                    nodo.uct=((nodo.wins*1.0)/(nodo.n*1.0))
                
                else:
                    nodo.uct=6.0

        '''          
        la lógica de este paso es:
        si el nodo no es nulo:

            si el padre del nodo no es nulo:
                si visitas distinto de cero:
                    nodo->uct=formula completa
                demás:
                    nodo->uct=0
            demás: 
                si visitas distinto de cero:
                    nodo->uct=promedio ponderado (solamente wins entre visitas)
                demás:
                    nodo->uct=0
        '''

    def seleccion(self, tipo_de_busqueda="BEST", exploracion=2):
        nodo_seleccionado=self.raiz

        if(tipo_de_busqueda=="BEST"):
            while nodo_seleccionado.hijos:
                for i in range(len(nodo_seleccionado.hijos)):
                    self.calcular_uct(nodo_seleccionado.hijos[i], exploracion)
                
                mejor=0

                for i in range(len(nodo_seleccionado.hijos)):
                    if(nodo_seleccionado.hijos[i].uct>nodo_seleccionado.hijos[mejor].uct):
                        mejor=i
                        
                nodo_seleccionado=nodo_seleccionado.hijos[mejor]

            return nodo_seleccionado
        
        elif(tipo_de_busqueda=="RANDOM"):
            while nodo_seleccionado.hijos:
                nodo_seleccionado=nodo_seleccionado.hijos[random.randint(0, len(nodo_seleccionado.hijos)-1)]

            return nodo_seleccionado
        
        elif(tipo_de_busqueda=="PROBABILISTIC"):
            while nodo_seleccionado.hijos:
                suma=0.0
                for i in range(len(nodo_seleccionado.hijos)):
                    self.calcular_uct(nodo_seleccionado.hijos[i], exploracion)
                    suma+=nodo_seleccionado.hijos[i].uct
                    
                rand=random.random()*suma

                elegido=0
                while(rand>=0):
                    rand-=nodo_seleccionado.hijos[elegido].uct
                    elegido+=1

                nodo_seleccionado=nodo_seleccionado.hijos[elegido]

            return nodo_seleccionado

    def expansion(self, nodo_seleccionado):
        tableros, turnos=self.juego.todos_los_tiros([fila.copy() for fila in nodo_seleccionado.tablero], 2 if nodo_seleccionado.turno==1 else 1)

        for i in range(len(tableros)):
            self.insertar(nodo_seleccionado, [fila.copy() for fila in tableros[i]], turnos, 0 if not self.juego.condiciones_de_terminado([fila.copy() for fila in tableros[i]]) else turnos)

        return nodo_seleccionado

    def simulacion(self, nodo_seleccionado, turnos, turno_copia):
        if(nodo_seleccionado.g_p==turno_copia):
            resultado=1
        
        elif(nodo_seleccionado.g_p!=0):
            resultado=0
        
        else:
            resultado=1 if turno_copia==self.juego.terminar_juego([fila.copy() for fila in nodo_seleccionado.tablero], turnos) else 0

        return resultado
    
    def back_prop(self, nodo_backprop, resultado):
        while(nodo_backprop.pad!=None):
            nodo_backprop.pad.wins+=resultado
            nodo_backprop.pad.n+=1
            nodo_backprop=nodo_backprop.pad

    def buscar_hasta_nivel(self, nodo, tablero, nivel_max, nivel_actual=0):
        if nodo is None:
            return None

        if nodo.tablero == tablero:
            return nodo

        if nivel_actual >= nivel_max:
            return None

        for hijo in nodo.hijos:
            encontrado = self.buscar_hasta_nivel(
                hijo,
                tablero,
                nivel_max,
                nivel_actual + 1
            )

            if encontrado is not None:
                return encontrado

        return None

    def MCTS(self, tablero, turno, tamano, exploracion=2, verbose=False):
        self.raiz=None

        turno_copia=turno
        
        if not self.juego.condiciones_de_terminado(tablero):
            if self.raiz==None:
                self.raiz=nodo([fila.copy() for fila in tablero], turno, 0)
                #a la raiz, darle todos los hijos que pueda tener, y darles uct
                tableros, turnos=self.juego.todos_los_tiros([fila.copy() for fila in tablero], turno)

                for i in range(len(tableros)):
                    nodo_backprop=self.insertar(self.raiz, 
                                                [fila.copy() for fila in tableros[i]],
                                                turnos, 
                                                0 if not self.juego.condiciones_de_terminado(tableros[i]) else 2 if turnos==1 else 1)
                    self.calcular_uct(nodo_backprop, exploracion)

                    self.calcular_uct(self.raiz, exploracion)

            else:
                self.raiz=self.buscar_hasta_nivel(self.raiz, tablero, 2)

            nodos_cantidad=0

            while(nodos_cantidad<=tamano):
                #SELECCION
                nodo_seleccionado=self.seleccion(tipo_de_busqueda="BEST", exploracion=2)

                if nodo_seleccionado==None:
                    #continue significa saltarse esta iteracion del while, entonces
                    #"si el nodo no existe, saltate a la siguiente"
                    continue

                if nodo_seleccionado.g_p==0:
                    #EXPANSION
                    nodo_seleccionado=self.expansion(nodo_seleccionado)

                    #SIMULACION
                    for nodo_hijo in nodo_seleccionado.hijos:
                        resultado=self.simulacion(nodo_hijo, turnos, turno_copia)
                        self.back_prop(nodo_hijo, resultado)

                else:
                    if(nodo_seleccionado.g_p==turno_copia):
                        resultado=1
                    
                    else:
                        resultado=0
                    
                    #BACKPROP
                    self.back_prop(nodo_seleccionado, resultado)
                        
                nodos_cantidad+=1
            
            if(verbose):
                self.fancy_print(self.raiz)
            
            mejor_juego=0
            for i in range(len(self.raiz.hijos)):
                if(self.raiz.hijos[i].uct>self.raiz.hijos[mejor_juego].uct):
                    mejor_juego=i
            
            tablero=[fila.copy() for fila in self.raiz.hijos[mejor_juego].tablero]
                
            return tablero, 2 if turno_copia==1 else 1
        
        else:
            if(verbose):
                print("NO_POSSIBLE_SHOT")
            
            return tablero,turno
        
    def leer_entero(self):
        while True:
            try:
                x = int(input())
                
                if x == 123456789:
                    sys.exit(2)
                
                return x

            except ValueError:
                print("Entrada inválida. Intenta de nuevo: ", end="")

    def leer_double(self):
        while True:
            try:
                x = float(input())
                
                if x == 123456789:
                    sys.exit(1)
                
                return x

            except ValueError:
                print("Entrada inválida. Intenta de nuevo: ", end="")

    def imprimir_marcador(self, turno):
        if(turno==1):
            print("O\n\n-------------------------------MARCADOR------------------------------\n", end="")
        else:
            print("X\n\n-------------------------------MARCADOR------------------------------\n", end="")
        print("╔═══════════════════════════════════════════════════════════════════╗\n", end="")
        print("║ Humano:      ", self.victoria_humana, "                                                  ║\n", end="")

        if(self.victoria_maquina!=0):
            print("║                             Razón:", (self.victoria_humana*1.0)/(self.victoria_maquina*1.0), "            Ventaja:", " HUMANO ║\n" if self.victoria_maquina<self.victoria_maquina else " MAQUINA ║\n" if self.victoria_maquina>self.victoria_maquina else " TABLAS ║\n", end="")
        else:
            print("║                             Razón:∞            Ventaja:", " HUMANO ║\n" if self.victoria_maquina<self.victoria_maquina else " MAQUINA ║\n" if self.victoria_maquina>self.victoria_maquina else " TABLAS ║\n", end="")
        
        print("║ Computadora: ", self.victoria_maquina, "                                                  ║\n", end="")
        print("╚═══════════════════════════════════════════════════════════════════╝\n", end="")
        print("\n\n\nQuieres seguir?:(1=si, 0=no)", end="")

    def imprimir_marcador_final(self):
        print("----------------------------MARCADOR FINAL---------------------------\n", end="")
        print("╔═══════════════════════════════════════════════════════════════════╗\n", end="")
        print("║ Humano:      ", self.victoria_humana, "                                                  ║\n", end="")
        
        if(self.victoria_maquina!=0):
            print("║                             Razón:", (self.victoria_humana*1.0)/(self.victoria_maquina*1.0), "            Ventaja:", " HUMANO ║\n" if self.victoria_maquina<self.victoria_maquina else " MAQUINA ║\n" if self.victoria_maquina>self.victoria_maquina else " TABLAS ║\n", end="")
        else:
            print("║                             Razón:∞            Ventaja:", " HUMANO ║\n" if self.victoria_maquina<self.victoria_maquina else " MAQUINA ║\n" if self.victoria_maquina>self.victoria_maquina else " TABLAS ║\n", end="")
            
        print("║ Computadora: ", self.victoria_maquina, "                                                  ║\n", end="")
        print("╚═══════════════════════════════════════════════════════════════════╝\n", end="")
        
        if(self.victoria_humana==self.victoria_maquina):
            print("-EMPATE\n\n", end="")
        
        elif(self.victoria_humana>self.victoria_maquina):
            print("-GANASTE\n\n", end="")
        
        else:
            print("-PERDISTE\n\n", end="")

    def _juego_contra_humano_inteligente(self, tamano, piezas_humano=1, inicio=1, verbose=0, dificultad=-1, tipo_de_busqueda=True):
        print("JUEGO CONTRA HUMANO:")
        tablero, turno=self.juego.poner_el_juego(tamano, inicio)

        seguir=1
        self.victoria_humana=0
        self.victoria_maquina=0
        self.juego.ver_tablero(tablero, turno)

        while(seguir==1):
            print("Turno de: ", "X" if turno==1 else "O")
            if(turno==piezas_humano):
                while(1):
                    print("\n\n\nFilas(origen):", end="")
                    f_o=self.leer_entero()
                    print(f_o, "\nColumnas(origen):", end="")
                    c_o=self.leer_entero()
                    print(c_o, "\nFilas(terminal):", end="")
                    f_t=self.leer_entero()
                    print(f_t, "\nColumnas(terminal):", end="")
                    c_t=self.leer_entero()
                    print(c_t)

                    if(f_o==-1 or c_o==-1 or f_t==-1 or c_t==-1):
                        seguir=0
                        break

                    if(self.juego.tiro_legal(f_o,c_o,f_t,c_t,tablero) and (tablero[f_o][c_o]==piezas_humano)):
                        print("\nTiro registrado")
                        tablero[f_o][c_o]=0
                        tablero[f_t][c_t]=piezas_humano

                        break
                    
                    else:
                        print("\nTiro no permitido, intenta de nuevo.", end="")

                print("Siguiendo con el juego")
                               
                turno=2 if piezas_humano==1 else 1
            
            else:
                print("\ntiro de maquina:\n", end="")
                print("nodos a calcular: ", int(dificultad*(0.02*pow(93.57, len(tablero)))) if dificultad!=-1 else "50000")

                tablero, turno=self.MCTS(tablero,
                                         turno, 
                                         50000 if dificultad==-1 else int(dificultad*(0.02*np.pow(93.57, len(tablero)))), 
                                         self.juego.siguiente_turno(turno), 
                                         verbose)
                
                self.juego.ver_tablero(tablero,turno)

                turno=piezas_humano

            if(self.juego.condiciones_de_terminado(tablero)):
                print("\nVICTORIA DE:", end="")
                if(turno!=piezas_humano):
                    self.victoria_humana+=1
                else:
                    self.victoria_maquina+=1

                self.imprimir_marcador(turno)
                
                seguir=int(input())
                if(seguir!=0):
                    seguir=1
                    tablero, turno=self.juego.poner_el_juego(tamano, inicio)
                    self.juego.ver_tablero(tablero, turno)
                
                else:
                    if(turno==1):
                        print("\nGracias por jugar, suerte para la próxima! :)\n", end="")
                    
                    else:
                        print("\nGracias por jugar, mejoraré para ganarte la próxima! >:)\n", end="")
            
        self.imprimir_marcador_final()
        
        
    def juego_contra_humano_inteligente(self):
        print("\n\nQuieres el juego default? (1=si, 0=entrar a configuracion)")
        setinds=self.leer_entero()
        if(setinds==0):
            print("\n\nQué piezas quieres tener? (X=1, O=2)  ")
            piezas=self.leer_entero()
            print("\n\nQuién quieres que inicie primero? (tu=0, computadora=1) ")
            primero=True if self.leer_entero()!=0 else False
            print("\n\nQué tamaño quieres que tenga el tablero?")
            tamanio=self.leer_entero()
            tamanio=3 if tamanio<3 else tamanio
            print("\n\nQuieres ver el arbol de probabilidad? (1=si, 0=no) (RECOMENDABLE NO VERLO, PUEDEN SER ARBOLES MUY GRANDES)")
            ver_=True if self.leer_entero()!=0 else False
            print("\n\nQué dificultad quieres para el juego? (valor entre 0 y 1, 0 es muy facil, 1 es muy dificil)")
            dificultad=self.leer_double()
            print("\n\nQué tipo de búsqueda quieres que se haga? (random=0, probabilistica=1) ")
            busq=True if self.leer_entero()!=0 else False
            self._juego_contra_humano_inteligente(tamanio, piezas, primero, ver_, dificultad, busq)
        
        else:
            self._juego_contra_humano_inteligente(3, 1, False, False, -1, True)

if __name__=="__main__":
    clasi=arbol(hexapawn())
    #clasi.juego_contra_humano_inteligente()
    clasi._juego_contra_humano_inteligente(4, 1, True, False, 0.1, False)

JUEGO CONTRA HUMANO:

╔═══════════════╗   filas     turno de:  X
║ O │ O │ O │ O ║  0
║---┼---┼---┼---║
║   │   │   │   ║  1
║---┼---┼---┼---║
║   │   │   │   ║  2
║---┼---┼---┼---║
║ X │ X │ X │ X ║  3
╚═══════════════╝
  0   1   2   3  

columnas


Turno de:  X



Filas(origen):3 
Columnas(origen):3 
Filas(terminal):2 
Columnas(terminal):3

Tiro registrado
Siguiendo con el juego
Turno de:  O

tiro de maquina:
nodos a calcular:  153312

╔═══════════════╗   filas     turno de:  X
║ O │ O │   │ O ║  0
║---┼---┼---┼---║
║   │   │ O │   ║  1
║---┼---┼---┼---║
║   │   │   │ X ║  2
║---┼---┼---┼---║
║ X │ X │ X │   ║  3
╚═══════════════╝
  0   1   2   3  

columnas


Turno de:  X



Filas(origen):3 
Columnas(origen):0 
Filas(terminal):2 
Columnas(terminal):0

Tiro registrado
Siguiendo con el juego
Turno de:  O

tiro de maquina:
nodos a calcular:  153312

╔═══════════════╗   filas     turno de:  X
║ O │ O │   │ O ║  0
║---┼---┼---┼---║
║   │   │   │   ║  1
║---┼---┼---┼---║
║ X │   │ O │ X ║